# 📘 NumPy for Data Engineering
## Databricks Notebook — Vectorized Numeric Processing

**What you'll learn:**
- Why NumPy is 100x faster than Python loops
- Array fundamentals: creation, indexing, reshaping
- Vectorized operations (the core power)
- Aggregation & statistics for DE
- DE applications: missing values, binning, time-series, batch processing
- Performance benchmarks: NumPy vs loops vs pandas
- NumPy + Spark integration

**Prerequisites:** Notebooks 01-09 (iot_manufacturing_dw from Notebook 06)

**Key rule:** NumPy is for numeric arrays on a single node. Use PySpark for distributed data.

---

---
# 📗 Section 1: Why NumPy for Data Engineering

**NumPy's superpower:** Vectorized operations on contiguous memory.
No Python loops, no type checking per element — just raw C-speed math.

| Use Case | NumPy | Python Loop | Speedup |
|---|---|---|---|
| Sum 1M numbers | 1ms | 100ms | 100x |
| Element-wise multiply | 2ms | 200ms | 100x |
| Statistical calculations | 3ms | 300ms | 100x |

**DE use cases:**
- Sensor data processing (IoT pipelines)
- Financial calculations (interest, risk scores)
- Statistical quality checks (mean, std, percentiles)
- Data generation for testing
- Batch normalization

In [ ]:
import numpy as np
import time

# Load sensor readings into NumPy
sensor_pdf = spark.table("iot_manufacturing_dw.sensor_readings").limit(50000).toPandas()
values = sensor_pdf["value"].values  # NumPy array
sensor_ids = sensor_pdf["sensor_id"].values

print(f"Loaded {len(values):,} sensor readings into NumPy")
print(f"Array type: {type(values)}")
print(f"Dtype: {values.dtype}")
print(f"Shape: {values.shape}")
print(f"Memory: {values.nbytes / 1024:.1f} KB")

# Basic stats (instant on 50K values)
print(f"\nStats: mean={values.mean():.2f}, std={values.std():.2f}, min={values.min():.2f}, max={values.max():.2f}")

---
# 📗 Section 2: Array Fundamentals

In [ ]:
# Creating arrays
zeros = np.zeros((3, 4))           # 3x4 of zeros
ones = np.ones(10)                  # 10 ones
range_arr = np.arange(0, 100, 5)   # 0 to 100, step 5
linspace = np.linspace(0, 1, 11)   # 11 evenly spaced from 0 to 1
full = np.full((2, 3), 42)         # 2x3 filled with 42

print(f"zeros shape: {zeros.shape}, dtype: {zeros.dtype}")
print(f"arange: {range_arr}")
print(f"linspace: {linspace}")
print(f"full:\n{full}")

In [ ]:
# Indexing and slicing
arr = np.arange(20).reshape(4, 5)
print(f"Array:\n{arr}")
print(f"\nRow 0: {arr[0]}")
print(f"Col 2: {arr[:, 2]}")
print(f"Subarray [1:3, 2:4]:\n{arr[1:3, 2:4]}")

# Boolean indexing (THE most useful for DE)
data = np.array([10, -5, 30, -2, 45, 0, 22, -8])
positive = data[data > 0]
print(f"\nOriginal: {data}")
print(f"Positive only: {positive}")

In [ ]:
# Simulate 24 hours of readings across 10 sensors
rng = np.random.default_rng(42)
# Shape: (24 hours, 10 sensors)
sensor_matrix = rng.normal(loc=50, scale=10, size=(24, 10))
print(f"Sensor matrix shape: {sensor_matrix.shape}")
print(f"Hour 0, all sensors: {sensor_matrix[0].round(1)}")
print(f"Sensor 0, all hours: {sensor_matrix[:, 0].round(1)}")

---
# 📗 Section 3: Vectorized Operations (The Core Power)

**Rule:** If you're writing a `for` loop over array elements, you're doing it wrong.
NumPy does it in C, 100x faster.

In [ ]:
# Vectorized arithmetic (no loops!)
temperatures_c = np.array([20.5, 22.1, 19.8, 25.3, 18.7, 23.4])

# Convert Celsius to Fahrenheit — entire array at once
temperatures_f = temperatures_c * 9/5 + 32
print(f"Celsius:    {temperatures_c}")
print(f"Fahrenheit: {temperatures_f.round(1)}")

# Comparison operations return boolean arrays
above_avg = temperatures_c > temperatures_c.mean()
print(f"Above average: {above_avg}")
print(f"Values above avg: {temperatures_c[above_avg]}")

In [ ]:
# Broadcasting: different shapes work together
# Sensor readings (5000 readings) - normalize to 0-1 range
readings = values[:5000].copy()
normalized = (readings - readings.min()) / (readings.max() - readings.min())
print(f"Original range: [{readings.min():.2f}, {readings.max():.2f}]")
print(f"Normalized range: [{normalized.min():.4f}, {normalized.max():.4f}]")

# Threshold detection (vectorized)
threshold = readings.mean() + 2 * readings.std()
anomalies = readings[readings > threshold]
print(f"\nThreshold (mean + 2σ): {threshold:.2f}")
print(f"Anomalies: {len(anomalies)} out of {len(readings)} ({len(anomalies)/len(readings)*100:.1f}%)")

In [ ]:
# ============================================
# 🎯 YOUR TURN: Temperature Anomalies
# ============================================
# Using the 'values' array (sensor readings):
# 1. Calculate the mean and std
# 2. Find values more than 3 standard deviations from the mean
# 3. Count how many anomalies there are
# 4. Calculate the anomaly rate (%)
# 5. Find the indices of the top 5 most extreme values
#    Hint: use np.argsort() or np.argpartition()
#
# Write your code below:


In [ ]:
# ============================================
# ✅ SOLUTION
# ============================================
mean = values.mean()
std = values.std()
z_scores = np.abs((values - mean) / std)

anomalies_mask = z_scores > 3
anomaly_count = anomalies_mask.sum()
anomaly_rate = anomaly_count / len(values) * 100

print(f"Mean: {mean:.2f}, Std: {std:.2f}")
print(f"Anomalies (>3σ): {anomaly_count} ({anomaly_rate:.2f}%)")

# Top 5 most extreme
top5_idx = np.argsort(z_scores)[-5:]
print(f"\nTop 5 extreme values:")
for idx in top5_idx:
    print(f"  Index {idx}: value={values[idx]:.2f}, z-score={z_scores[idx]:.2f}")

---
# 📗 Section 4: Aggregation & Statistics for DE

In [ ]:
# Axis-based operations
# Reshape: 100 sensors x 500 readings each
matrix = values[:50000].reshape(100, 500)
print(f"Matrix shape: {matrix.shape} (100 sensors x 500 readings)")

# Per-sensor stats (axis=1 = across columns = per row)
sensor_means = matrix.mean(axis=1)
sensor_stds = matrix.std(axis=1)
print(f"\nPer-sensor means (first 10): {sensor_means[:10].round(2)}")
print(f"Per-sensor stds (first 10): {sensor_stds[:10].round(2)}")

# Per-time-slot stats (axis=0 = across rows = per column)
time_means = matrix.mean(axis=0)
print(f"\nPer-time-slot means (first 10): {time_means[:10].round(2)}")

In [ ]:
# Cumulative operations (running totals — common in finance)
daily_revenue = np.array([1500, 2200, 1800, 3100, 2500, 1900, 2800])
cumulative = np.cumsum(daily_revenue)
print(f"Daily revenue: {daily_revenue}")
print(f"Cumulative:    {cumulative}")
print(f"Running avg:   {cumulative / np.arange(1, len(daily_revenue)+1)}")

# Percentiles
print(f"\nSensor value percentiles:")
for p in [25, 50, 75, 90, 95, 99]:
    print(f"  P{p}: {np.percentile(values, p):.2f}")

---
# 📗 Section 5: Data Engineering Applications

## 5.1 Handling Missing Values

In [ ]:
# Simulate missing values
data_with_gaps = values[:100].copy().astype(float)
# Introduce NaN gaps
gap_indices = [5, 6, 7, 20, 21, 50, 51, 52, 53]
data_with_gaps[gap_indices] = np.nan

print(f"Total values: {len(data_with_gaps)}")
print(f"NaN count: {np.isnan(data_with_gaps).sum()}")

# NaN-safe statistics
print(f"Mean (with NaN): {np.mean(data_with_gaps)}")  # Returns NaN!
print(f"Mean (NaN-safe): {np.nanmean(data_with_gaps):.2f}")
print(f"Std (NaN-safe): {np.nanstd(data_with_gaps):.2f}")

# Interpolation to fill gaps
x = np.arange(len(data_with_gaps))
valid_mask = ~np.isnan(data_with_gaps)
filled = np.interp(x, x[valid_mask], data_with_gaps[valid_mask])
print(f"\nAfter interpolation — NaN count: {np.isnan(filled).sum()}")

## 5.2 Binning & Bucketing

In [ ]:
# np.digitize: assign values to bins
amounts = values[:1000]
bin_edges = [0, 20, 40, 60, 80, 100]
bin_labels = ["Very Low", "Low", "Medium", "High", "Very High"]

# Digitize returns bin indices (1-based)
bin_indices = np.digitize(amounts, bin_edges)
print(f"Bin edges: {bin_edges}")
print(f"First 10 values: {amounts[:10].round(1)}")
print(f"First 10 bins:   {bin_indices[:10]}")

# Count per bin
unique, counts = np.unique(bin_indices, return_counts=True)
print(f"\nDistribution:")
for u, c in zip(unique, counts):
    label = bin_labels[u-1] if u <= len(bin_labels) else "Above Max"
    print(f"  Bin {u} ({label}): {c} values ({c/len(amounts)*100:.1f}%)")

## 5.3 Time-Series: Detecting Spikes

In [ ]:
# np.diff: differences between consecutive values
readings = values[:200]
diffs = np.diff(readings)

# Detect sudden changes (spikes)
spike_threshold = 3 * np.std(diffs)
spikes = np.where(np.abs(diffs) > spike_threshold)[0]

print(f"Readings: {len(readings)}")
print(f"Spike threshold: {spike_threshold:.2f}")
print(f"Spikes detected: {len(spikes)}")
if len(spikes) > 0:
    print(f"Spike locations: {spikes[:10]}")
    print(f"Spike magnitudes: {diffs[spikes[:5]].round(2)}")

## 5.4 Random Data Generation (for testing)

In [ ]:
# Modern random generator (reproducible)
rng = np.random.default_rng(seed=42)

# Generate realistic test data
n = 1000
test_data = {
    "transaction_amounts": rng.lognormal(mean=4, sigma=1.5, size=n).round(2),
    "customer_ages": rng.integers(18, 80, size=n),
    "risk_scores": rng.normal(50, 15, size=n).clip(0, 100).round(1),
    "categories": rng.choice(["A", "B", "C", "D"], size=n, p=[0.4, 0.3, 0.2, 0.1]),
}

print("Generated test data:")
for name, arr in test_data.items():
    if arr.dtype.kind in ('f', 'i'):
        print(f"  {name}: mean={arr.mean():.1f}, std={arr.std():.1f}, range=[{arr.min():.1f}, {arr.max():.1f}]")
    else:
        unique, counts = np.unique(arr, return_counts=True)
        print(f"  {name}: {dict(zip(unique, counts))}")

---
# 📗 Section 6: Performance Comparison

In [ ]:
import time

# Benchmark: sum 1 million numbers
n = 1_000_000
py_list = list(range(n))
np_arr = np.arange(n)

# Python loop
start = time.time()
total = sum(py_list)
py_time = time.time() - start

# NumPy
start = time.time()
total = np_arr.sum()
np_time = time.time() - start

print(f"Sum of {n:,} numbers:")
print(f"  Python loop: {py_time*1000:.2f} ms")
print(f"  NumPy:       {np_time*1000:.2f} ms")
print(f"  Speedup:     {py_time/np_time:.0f}x")

In [ ]:
# Benchmark: element-wise operation
import pandas as pd

n = 500_000
np_arr = np.random.default_rng(42).normal(50, 10, n)
pd_series = pd.Series(np_arr)

# NumPy: normalize
start = time.time()
np_result = (np_arr - np_arr.mean()) / np_arr.std()
np_time = time.time() - start

# Pandas: normalize
start = time.time()
pd_result = (pd_series - pd_series.mean()) / pd_series.std()
pd_time = time.time() - start

# Python loop: normalize
start = time.time()
mean = sum(np_arr) / len(np_arr)
std = (sum((x - mean)**2 for x in np_arr) / len(np_arr))**0.5
loop_result = [(x - mean) / std for x in np_arr]
loop_time = time.time() - start

print(f"Normalize {n:,} values:")
print(f"  Python loop: {loop_time*1000:.1f} ms")
print(f"  Pandas:      {pd_time*1000:.1f} ms")
print(f"  NumPy:       {np_time*1000:.1f} ms")
print(f"  NumPy vs loop: {loop_time/np_time:.0f}x faster")
print(f"  NumPy vs pandas: {pd_time/np_time:.1f}x faster")

In [ ]:
# Memory comparison
import sys

n = 100_000
py_list = list(range(n))
np_arr = np.arange(n, dtype=np.int64)

py_mem = sys.getsizeof(py_list) + sum(sys.getsizeof(x) for x in py_list[:100]) * (n // 100)
np_mem = np_arr.nbytes

print(f"Memory for {n:,} integers:")
print(f"  Python list: ~{py_mem/1024/1024:.1f} MB")
print(f"  NumPy array: {np_mem/1024/1024:.1f} MB")
print(f"  Savings: {(1-np_mem/py_mem)*100:.0f}%")

---
# 📗 Section 7: NumPy + Spark Integration

In [ ]:
# Spark → NumPy → Spark roundtrip
import pandas as pd

# Get data from Spark
pdf = spark.table("iot_manufacturing_dw.sensor_readings").limit(10000).toPandas()
values_np = pdf["value"].values  # NumPy array

# Fast NumPy calculations
z_scores = np.abs((values_np - values_np.mean()) / values_np.std())
is_anomaly = z_scores > 2.5

# Add results back to pandas, then to Spark
pdf["z_score"] = z_scores
pdf["np_anomaly"] = is_anomaly

# Write back to Spark
result_df = spark.createDataFrame(pdf)
result_df.write.format("delta").mode("overwrite").saveAsTable("iot_manufacturing_dw.readings_with_zscore")
print(f"Written {result_df.count():,} rows with z-scores")
print(f"Anomalies detected: {is_anomaly.sum()}")

---
# 🚀 Section 8: Mini Projects

## Project 1: Sensor Anomaly Detection System

In [ ]:
# Complete anomaly detection using NumPy
def detect_anomalies_numpy(values, window=100, threshold=3.0):
    """Detect anomalies using rolling z-score with NumPy.
    
    Args:
        values: NumPy array of sensor readings
        window: rolling window size
        threshold: z-score threshold for anomaly
    
    Returns:
        dict with anomaly indices, scores, and summary
    """
    n = len(values)
    z_scores = np.zeros(n)
    
    # Calculate rolling mean and std using cumsum trick (fast!)
    cumsum = np.cumsum(np.insert(values, 0, 0))
    cumsum_sq = np.cumsum(np.insert(values**2, 0, 0))
    
    for i in range(window, n):
        w_sum = cumsum[i+1] - cumsum[i+1-window]
        w_sum_sq = cumsum_sq[i+1] - cumsum_sq[i+1-window]
        w_mean = w_sum / window
        w_std = np.sqrt(w_sum_sq / window - w_mean**2)
        if w_std > 0:
            z_scores[i] = abs(values[i] - w_mean) / w_std
    
    anomaly_mask = z_scores > threshold
    
    return {
        "total_readings": n,
        "anomaly_count": anomaly_mask.sum(),
        "anomaly_rate": anomaly_mask.sum() / n * 100,
        "anomaly_indices": np.where(anomaly_mask)[0],
        "max_z_score": z_scores.max(),
        "z_scores": z_scores
    }

# Run on sensor data
sensor_values = sensor_pdf["value"].values.astype(float)
result = detect_anomalies_numpy(sensor_values, window=100, threshold=3.0)

print(f"Anomaly Detection Results:")
print(f"  Total readings: {result['total_readings']:,}")
print(f"  Anomalies: {result['anomaly_count']:,} ({result['anomaly_rate']:.2f}%)")
print(f"  Max z-score: {result['max_z_score']:.2f}")

## Project 2: Financial Risk Calculator

In [ ]:
# Risk scoring with NumPy
# Load transaction data
txn_pdf = spark.table("iot_manufacturing_dw.sensor_readings").limit(20000).toPandas()
amounts = txn_pdf["value"].values.astype(float)
accounts = txn_pdf["sensor_id"].values  # using sensor_id as account proxy

# Per-account statistics
unique_accounts = np.unique(accounts)
risk_scores = np.zeros(len(unique_accounts))

for i, acc in enumerate(unique_accounts):
    acc_amounts = amounts[accounts == acc]
    if len(acc_amounts) > 5:
        # Risk factors
        volatility = acc_amounts.std() / max(acc_amounts.mean(), 1)
        max_ratio = acc_amounts.max() / max(acc_amounts.mean(), 1)
        spike_count = np.sum(np.abs(np.diff(acc_amounts)) > 2 * acc_amounts.std())
        
        risk_scores[i] = (volatility * 30 + max_ratio * 20 + spike_count * 5)

# Bucket into tiers
tier_edges = [0, 30, 60, 90, np.inf]
tier_labels = ["Low", "Medium", "High", "Critical"]
tiers = np.digitize(risk_scores, tier_edges)

print("Risk Score Distribution:")
for t in range(1, 5):
    count = (tiers == t).sum()
    print(f"  {tier_labels[t-1]}: {count} accounts ({count/len(tiers)*100:.1f}%)")

## Project 3: Fast Data Profiler

In [ ]:
def numpy_profile(arr, name="column"):
    """Fast data profiling using NumPy."""
    clean = arr[~np.isnan(arr)] if arr.dtype.kind == 'f' else arr[arr != None]
    
    profile = {
        "column": name,
        "count": len(arr),
        "null_count": len(arr) - len(clean),
        "null_pct": (len(arr) - len(clean)) / len(arr) * 100,
        "unique": len(np.unique(clean)),
        "min": float(np.min(clean)),
        "max": float(np.max(clean)),
        "mean": float(np.mean(clean)),
        "std": float(np.std(clean)),
        "p25": float(np.percentile(clean, 25)),
        "p50": float(np.percentile(clean, 50)),
        "p75": float(np.percentile(clean, 75)),
    }
    
    # Outliers (IQR method)
    iqr = profile["p75"] - profile["p25"]
    lower = profile["p25"] - 1.5 * iqr
    upper = profile["p75"] + 1.5 * iqr
    profile["outliers"] = int(np.sum((clean < lower) | (clean > upper)))
    
    return profile

# Profile sensor values
result = numpy_profile(values.astype(float), "sensor_value")
print("NumPy Profile:")
for k, v in result.items():
    if isinstance(v, float):
        print(f"  {k}: {v:.2f}")
    else:
        print(f"  {k}: {v}")

---
# 🏆 Section 9: Interview Questions

## Q1: Why is NumPy faster than Python lists?

1. **Contiguous memory:** Arrays stored in one block (cache-friendly)
2. **No type checking:** All elements same type (no per-element overhead)
3. **Vectorized C code:** Operations compiled in C/Fortran, not interpreted Python
4. **No Python object overhead:** int64 = 8 bytes vs Python int = 28+ bytes

## Q2: Explain broadcasting

Broadcasting lets NumPy operate on arrays of different shapes:
- `(5,3) + (3,)` → broadcasts (3,) to (5,3) — adds to each row
- `(5,1) + (1,3)` → broadcasts both to (5,3)
- **Rule:** Dimensions must be equal OR one of them must be 1
- **Fails:** `(5,3) + (4,)` — incompatible shapes

## Q3: NumPy vs pandas vs PySpark?

- **NumPy:** Pure numeric arrays, < 1GB, simple math/stats, maximum speed
- **Pandas:** Mixed types, labeled data, complex groupby, < 1GB
- **PySpark:** Any size, distributed, production pipelines, complex joins

## Q4: Missing values in NumPy?

- NumPy uses `np.nan` (only for float arrays — int arrays can't have NaN)
- Detection: `np.isnan(arr)`
- Safe stats: `np.nanmean()`, `np.nanstd()`, `np.nansum()`
- Fill: `np.interp()` for interpolation, or `arr[np.isnan(arr)] = fill_value`

## Q5: View vs copy?

- **View:** Shares memory with original. Changes to view affect original.
  - Created by: slicing (`arr[1:5]`), reshape, transpose
- **Copy:** Independent memory. Changes don't affect original.
  - Created by: `.copy()`, fancy indexing (`arr[[1,3,5]]`), boolean indexing
- **Bug risk:** Modifying a view accidentally changes the source array

## Q6: Anomaly detection with NumPy?

1. Calculate rolling mean and std (using cumsum trick for O(n) speed)
2. Compute z-score: `(value - rolling_mean) / rolling_std`
3. Flag anomalies: `z_score > threshold` (typically 2.5-3.0)
4. Optionally: use IQR method as alternative (P75-P25, flag > 1.5*IQR)

---
# 📗 Section 6: Advanced Array Operations for DE

NumPy's power comes from vectorized operations — replacing Python loops
with C-speed array math. These patterns appear constantly in DE work.

In [ ]:
import numpy as np

# --- Structured arrays (like a lightweight table) ---
dtype = np.dtype([
    ("order_id", "i4"),
    ("amount", "f8"),
    ("status", "U10"),
    ("customer_id", "i4"),
])
orders = np.array([
    (1, 150.0, "completed", 101),
    (2, 200.0, "pending",   102),
    (3, 75.0,  "cancelled", 101),
    (4, 300.0, "completed", 103),
    (5, 50.0,  "completed", 102),
], dtype=dtype)

# Access like a DataFrame
print("All amounts:", orders["amount"])
print("Completed orders:", orders[orders["status"] == "completed"])
print("Total completed revenue:", orders[orders["status"] == "completed"]["amount"].sum())

In [ ]:
# --- np.where: vectorized if/else ---
amounts = np.array([150.0, 200.0, 75.0, 300.0, 50.0, 1500.0, 25.0])

# Categorize orders
categories = np.where(amounts >= 500, "high",
             np.where(amounts >= 100, "medium", "low"))
print("Categories:", categories)

# Apply tiered discount
discounts = np.where(amounts >= 500, 0.15,
            np.where(amounts >= 100, 0.10, 0.05))
final_amounts = amounts * (1 - discounts)
print("After discount:", np.round(final_amounts, 2))

# --- np.select: multi-condition (like SQL CASE WHEN) ---
conditions = [amounts >= 1000, amounts >= 500, amounts >= 100]
choices = ["platinum", "gold", "silver"]
tiers = np.select(conditions, choices, default="bronze")
print("Tiers:", tiers)

In [ ]:
# --- Vectorized string operations with numpy ---
# For large arrays, numpy char operations can be faster than Python loops
emails = np.array([
    "user@example.com",
    "ADMIN@COMPANY.ORG",
    "test.user@domain.co.uk",
    "invalid-email",
    "another@test.com",
])

# Extract domains
domains = np.char.split(emails, "@")
# domains is an array of lists — use vectorized approach
valid_mask = np.array(["@" in e for e in emails])
print("Valid emails:", emails[valid_mask])
print("Invalid emails:", emails[~valid_mask])

# Lowercase all
lowered = np.char.lower(emails)
print("Lowercased:", lowered)

## 6.2 Statistical Operations for Data Quality

NumPy provides fast statistical functions essential for data profiling
and anomaly detection in DE pipelines.

In [ ]:
# --- Z-score based outlier detection ---
def detect_outliers_zscore(arr, threshold=3.0):
    """Flag values more than `threshold` standard deviations from mean."""
    mean = np.mean(arr)
    std = np.std(arr)
    z_scores = np.abs((arr - mean) / std)
    outlier_mask = z_scores > threshold
    return outlier_mask, z_scores

# Simulate sensor readings with outliers
np.random.seed(42)
readings = np.concatenate([
    np.random.normal(100, 10, 95),  # Normal readings
    np.array([500, -200, 800, 0, 999])  # Outliers
])

outlier_mask, z_scores = detect_outliers_zscore(readings)
print(f"Total readings: {len(readings)}")
print(f"Outliers detected: {outlier_mask.sum()}")
print(f"Outlier values: {readings[outlier_mask]}")
print(f"Outlier z-scores: {z_scores[outlier_mask].round(1)}")

# Clean data (remove outliers)
clean_readings = readings[~outlier_mask]
print(f"\nClean data: {len(clean_readings)} readings")
print(f"Mean: {clean_readings.mean():.1f}, Std: {clean_readings.std():.1f}")

In [ ]:
# --- IQR-based outlier detection (more robust) ---
def detect_outliers_iqr(arr, multiplier=1.5):
    """Tukey's fence method — robust to extreme outliers."""
    q1 = np.percentile(arr, 25)
    q3 = np.percentile(arr, 75)
    iqr = q3 - q1
    lower = q1 - multiplier * iqr
    upper = q3 + multiplier * iqr
    outlier_mask = (arr < lower) | (arr > upper)
    return outlier_mask, lower, upper

outlier_mask, lower, upper = detect_outliers_iqr(readings)
print(f"IQR bounds: [{lower:.1f}, {upper:.1f}]")
print(f"Outliers (IQR): {outlier_mask.sum()}")
print(f"Outlier values: {readings[outlier_mask]}")

# --- Percentile-based capping (winsorization) ---
def winsorize(arr, lower_pct=1, upper_pct=99):
    """Cap extreme values at percentiles instead of removing them."""
    lower = np.percentile(arr, lower_pct)
    upper = np.percentile(arr, upper_pct)
    return np.clip(arr, lower, upper)

capped = winsorize(readings)
print(f"\nAfter winsorization: min={capped.min():.1f}, max={capped.max():.1f}")

---
# 📗 Section 7: NumPy for ETL Performance

These patterns show how NumPy accelerates common ETL operations.

In [ ]:
# --- Fast hash-based deduplication ---
import hashlib

def fast_dedup_numpy(records):
    """
    Deduplicate records using numpy for speed.
    Much faster than pandas drop_duplicates for simple cases.
    """
    # Convert to structured array
    hashes = np.array([hash(str(r)) for r in records])
    _, unique_indices = np.unique(hashes, return_index=True)
    unique_indices.sort()  # Preserve order
    return [records[i] for i in unique_indices]

# Test
records = [
    {"id": 1, "val": "a"},
    {"id": 2, "val": "b"},
    {"id": 1, "val": "a"},  # duplicate
    {"id": 3, "val": "c"},
    {"id": 2, "val": "b"},  # duplicate
]
deduped = fast_dedup_numpy(records)
print(f"Original: {len(records)}, Deduped: {len(deduped)}")

# --- Vectorized type conversion ---
raw_strings = np.array(["1.5", "2.3", "bad", "4.1", "N/A", "3.7"])
# Safe conversion with error handling
def safe_float_convert(arr):
    result = np.full(len(arr), np.nan)
    for i, val in enumerate(arr):
        try:
            result[i] = float(val)
        except (ValueError, TypeError):
            pass  # Leave as NaN
    return result

converted = safe_float_convert(raw_strings)
print(f"\nConverted: {converted}")
print(f"Valid values: {converted[~np.isnan(converted)]}")
print(f"Failed conversions: {np.isnan(converted).sum()}")

In [ ]:
# --- Benchmark: NumPy vs Python loops ---
import time

n = 1_000_000
data = np.random.uniform(0, 1000, n)

# Python loop
start = time.time()
result_loop = [x * 1.1 if x > 500 else x * 0.9 for x in data]
loop_time = time.time() - start

# NumPy vectorized
start = time.time()
result_numpy = np.where(data > 500, data * 1.1, data * 0.9)
numpy_time = time.time() - start

print(f"Python loop: {loop_time:.3f}s")
print(f"NumPy vectorized: {numpy_time:.4f}s")
print(f"Speedup: {loop_time / numpy_time:.0f}x faster")
print(f"Results match: {np.allclose(result_loop, result_numpy)}")

---
# 📗 Section 8: Practice Exercises

In [ ]:
# ============================================================
# EXERCISE 1: Sensor Data Quality Pipeline
# ============================================================
# Given sensor readings, implement:
# 1. Detect and count outliers (z-score > 3)
# 2. Replace outliers with the column median
# 3. Normalize to [0, 1] range
# 4. Compute rolling 5-point average

np.random.seed(7)
sensors = np.random.normal(50, 10, (100, 4))  # 100 readings, 4 sensors
# Inject outliers
sensors[10, 0] = 500
sensors[50, 2] = -300
sensors[75, 1] = 999

def clean_sensor_data(data):
    result = data.copy()
    for col in range(data.shape[1]):
        col_data = data[:, col]
        mean, std = col_data.mean(), col_data.std()
        z_scores = np.abs((col_data - mean) / std)
        outlier_mask = z_scores > 3
        
        if outlier_mask.any():
            median = np.median(col_data[~outlier_mask])
            result[outlier_mask, col] = median
            print(f"  Sensor {col}: replaced {outlier_mask.sum()} outliers with median={median:.1f}")
    
    # Normalize each column to [0, 1]
    col_min = result.min(axis=0)
    col_max = result.max(axis=0)
    normalized = (result - col_min) / (col_max - col_min)
    
    # Rolling 5-point average (manual implementation)
    rolling_avg = np.zeros_like(normalized)
    for i in range(len(normalized)):
        start = max(0, i - 4)
        rolling_avg[i] = normalized[start:i+1].mean(axis=0)
    
    return normalized, rolling_avg

normalized, rolling = clean_sensor_data(sensors)
print(f"\nNormalized range: [{normalized.min():.3f}, {normalized.max():.3f}]")
print(f"Rolling avg shape: {rolling.shape}")
print("First 5 rows (normalized):")
print(np.round(normalized[:5], 3))

In [ ]:
# ============================================================
# EXERCISE 2: Financial Risk Calculator
# ============================================================
# Compute Value at Risk (VaR) and portfolio statistics

np.random.seed(42)
# Daily returns for 5 assets over 252 trading days
returns = np.random.normal(0.001, 0.02, (252, 5))
weights = np.array([0.3, 0.25, 0.2, 0.15, 0.1])  # Portfolio weights

# Portfolio daily returns
portfolio_returns = returns @ weights

# Statistics
print("Portfolio Statistics:")
print(f"  Mean daily return:  {portfolio_returns.mean():.4f} ({portfolio_returns.mean()*252:.2%} annualized)")
print(f"  Daily volatility:   {portfolio_returns.std():.4f} ({portfolio_returns.std()*np.sqrt(252):.2%} annualized)")
print(f"  Sharpe ratio:       {(portfolio_returns.mean() / portfolio_returns.std() * np.sqrt(252)):.2f}")

# Value at Risk (VaR) — 95% and 99% confidence
var_95 = np.percentile(portfolio_returns, 5)
var_99 = np.percentile(portfolio_returns, 1)
print(f"  VaR (95%):          {var_95:.4f} (lose more than this 5% of days)")
print(f"  VaR (99%):          {var_99:.4f} (lose more than this 1% of days)")

# Correlation matrix
corr_matrix = np.corrcoef(returns.T)
print(f"\nCorrelation matrix (5 assets):")
print(np.round(corr_matrix, 2))

---
# ✅ Notebook Complete!

**What was covered:**
- NumPy fundamentals: arrays, dtypes, indexing, reshaping
- Vectorized operations: arithmetic, comparisons, boolean indexing, broadcasting
- Aggregation: axis-based stats, cumulative, percentiles
- DE applications: missing values, binning, spike detection, data generation
- Performance: 100x speedup over loops, memory savings
- Spark integration: toPandas → NumPy → createDataFrame
- 3 mini projects: Anomaly Detection, Risk Calculator, Fast Profiler
- 6 interview questions

**Next:** Notebook 11 — Pydantic for Data Engineering

---